# 6.1a 原始数据探索与冻结数据契约

1. 检查原始 CTIS 与 ChMusic 本地副本的类别、文件和跨 split 重叠情况；
2. 读取本章主实验使用的 CTMP 冻结 manifest，核对六类任务、切片和数据划分契约。

前半部分生成的 `outputs/top10_class_list.json` 只记录原始 CTIS 的探索性统计，
当前 6.1b--6.4 主实验均不读取该文件，而是统一读取 `_data_pipeline/output/`。


## 1. 环境准备


In [ ]:
import sys
from pathlib import Path

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录；PROJECT_ROOT 指向 CODE/
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
PROJECT_ROOT = _p / "CODE"  # CODE/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)


### 1.1 依赖与数据齐备性检查

`require_ctis_cache=True` 只做提示——若 `CODE/datasets/CCMUSIC_CTIS/default/` 不存在，
后续第一个 `load_from_disk` 会触发一次性下载（约 600 MB，需联网）。


In [ ]:
from chapter06._common import check_environment

check_environment(notebook='06_1a', require_ctis_cache=True, raise_on_error=False)


In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import Audio, load_from_disk

from chapter06._common import setup_chinese_font
from chapter06.traditional_ml.data_loading import (
    DEFAULT_LOCAL_CACHE,
    compute_audio_sha1,
    get_top_n_classes,
)

setup_chinese_font()

OUT_DIR = PROJECT_ROOT / 'chapter06' / 'traditional_ml' / 'outputs'
OUT_DIR.mkdir(exist_ok=True)
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)


## 2. 加载 default/train 元数据

CTIS 共有 4 个 split：`default/train`、`eval/train`、`eval/validation`、`eval/test`。
其中**只有 `default/train` 含嵌入式音频字节**；eval 三 split 只存预算特征（mel/cqt/chroma）。
因此 6.1 的传统机器学习管线只能用 `default/train`。


In [ ]:
ds = load_from_disk(str(DEFAULT_LOCAL_CACHE))['train']
print('default/train rows:', len(ds))
print('columns:', ds.column_names)


## 3. 类别长尾分布

把所有类按样本数降序排列，画长尾柱状图。标注样本数 ≥ 50 的阈值线——
我们待会儿要论证「N=10 不是拍脑袋，恰是 `default_train ≥ 50` 的类别个数」。


In [ ]:
counter = Counter(ds['label'])
cname_map: dict[int, str] = {}
for label_idx, cname in zip(ds['label'], ds['cname']):
    cname_map.setdefault(label_idx, cname)

ranked = sorted(counter.items(), key=lambda kv: (-kv[1], kv[0]))
counts = np.array([c for _, c in ranked])

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(counts)), counts, color='0.35', width=1.0)
ax.axhline(50, color='black', linestyle='--', linewidth=1.0, label='样本数 = 50')
ax.set_xlabel('类别（按样本数降序）')
ax.set_ylabel('样本数')
ax.set_title(f'CTIS default/train 类别长尾分布（共 {len(ranked)} 类）')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'class_long_tail.png', dpi=600, bbox_inches='tight')
plt.show()

n_ge_50 = sum(1 for _, c in ranked if c >= 50)
print(f'总样本数: {len(ds)}，类别总数: {len(ranked)}')
print(f'样本数 ≥ 50 的类别数: {n_ge_50}')
print(f'前 10 类合计样本数: {counts[:10].sum()}')

# 数据版本变化时在此显式失败，提醒同步更新小结文字。
expected = (4956, 219, 10, 598)  # (总样本数, 类别总数, ≥50 类别数, 前 10 类合计)
actual = (len(ds), len(ranked), n_ge_50, int(counts[:10].sum()))
if actual != expected:
    raise RuntimeError(
        f'CTIS default/train 统计与第 7 节小结不一致：{actual} != {expected}，'
        '请先核对数据版本，再同步更新小结与本处的期望值'
    )


## 4. 原始 CTIS 的 top-10 统计快照

下表列出 `default/train` 中样本数最多的10个原始类别。当前本章任务的六个族级类别
由 `freeze_config.yaml` 的白名单、外部类别覆盖和质量控制规则共同确定，并不由这张
top-10表自动推出。


In [ ]:
top10 = get_top_n_classes(n=10)
df_top10 = pd.DataFrame(top10)
df_top10.insert(0, 'rank', range(1, len(df_top10) + 1))
df_top10


## 5. 跨 split 重叠检测

CTIS eval 三 split 不含音频字节，但其元数据 CSV（作者预先用 `export_ctis_metadata.py`
导出）里有 `audio_sha1` 列。我们对 `default/train` 的音频字节做 SHA1，再与 eval 三 split
的 `audio_sha1` 集合求交集，验证是否存在跨 split 样本重叠。

> 若读者本地没有元数据 CSV，可跳过本节；这部分仅作"数据集尽职调查"展示，不影响后续训练。


In [ ]:
ds_bytes = ds.cast_column('audio', Audio(decode=False))
default_sha1 = {compute_audio_sha1(row['audio']['bytes']) for row in ds_bytes}
print(f'default/train 唯一 SHA1 数量: {len(default_sha1)}  (总样本数 {len(ds_bytes)})')

meta_csv_dir = PROJECT_ROOT / 'chapter06' / '_data_pipeline' / 'preprocess' / 'metadata_export'
overlap_table = []
if meta_csv_dir.exists():
    eval_meta = pd.read_csv(meta_csv_dir / 'ctis_metadata_eval.csv')
    for split_name, group in eval_meta.groupby('split_name'):
        eval_sha1 = set(group['audio_sha1'].dropna())
        overlap = default_sha1 & eval_sha1
        overlap_table.append({
            'eval_split': split_name,
            'eval_samples': len(group),
            'overlap_with_default_train': len(overlap),
        })
    df_overlap = pd.DataFrame(overlap_table)
    display(df_overlap)
else:
    print('元数据 CSV 不存在，跳过跨 split 重叠检测。')


## 6. 保存原始 top-10 统计

该JSON用于保留前述原始数据统计。当前主实验不读取它。


In [ ]:
out_path = OUT_DIR / 'top10_class_list.json'
out_path.write_text(
    json.dumps({'classes': top10}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('written:', out_path.resolve())
print(json.dumps({'classes': top10[:3]}, ensure_ascii=False, indent=2), '...')


## 7. 原始 CTIS 探索小结

- CTIS `default/train` 共 4956 样本、219 类，呈现明显长尾（统计数字由第 3 节的契约核对绑定，数据版本变化会显式报错）。
- 样本数不少于50的10个类别合计598个样本；这只是原始标签层级的统计快照。
- 跨 split 重叠（若有）由 SHA1 检测客观给出，作为数据集质量的事实记录。


## 8. ChMusic 本地副本概览

同一数据来源内的划分不能代替独立来源上的评估。本地 ChMusic 副本按文件名前缀
覆盖11个乐器标签，每个标签有5个wav文件；副本未附足以核对演奏者、麦克风或房间的
元数据，因此这里不对这些因素作具体断言。

本节只检查旧版3秒探索切片。当前主实验使用清洗管线生成的24 kHz、5秒
`external_test`切片，见第9节。


In [ ]:
from chapter06._common.chmusic_loader import (
    CHMUSIC_LABEL_MAP,
    CTIS_INTERSECTION,
    load_chmusic_clips,
)

# 加载全部11类旧版3秒探索切片；不作为当前主实验输入
chm_all = load_chmusic_clips(intersection_only=False)
print(f'ChMusic 共加载 {len(chm_all)} 个 3 秒切片')
print(f'  覆盖 {len({c["cname"] for c in chm_all})} 类，{len({c["source_file"] for c in chm_all})} 段曲目')


### 8.1 ChMusic 11 类构成

每类 5 段曲目切出的 3 秒切片数不等（曲目长度有差异）。


In [ ]:
from collections import Counter

clip_counts = Counter([c['cname'] for c in chm_all])
order_11 = [CHMUSIC_LABEL_MAP[i] for i in range(1, 12)]
values = [clip_counts[n] for n in order_11]
in_intersection = [n in CTIS_INTERSECTION for n in order_11]
colors = ['0.25' if hit else '0.70' for hit in in_intersection]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(order_11)), values, color=colors)
ax.set_xticks(range(len(order_11)))
ax.set_xticklabels(order_11, rotation=30, ha='right')
ax.set_ylabel('3 秒切片数')
ax.set_title('ChMusic 11 类切片数（深色 = 与 CTIS top-10 严格交集）')
fig.tight_layout()
fig.savefig(FIG_DIR / 'chmusic_class_counts.png', dpi=600, bbox_inches='tight')
plt.show()


### 8.2 与 CTIS top-10 的类名对齐

ChMusic 和 CTIS 都按中文乐器名分类，但**两个数据集在分类粒度和命名规范上并不完全
一致**。下表把 CTIS top-10 与 ChMusic 11 类按门类做对齐，明确哪些可以直接拿来
做外部测试，哪些不行。

- **严格交集**：CTIS 与 ChMusic 都有，且为同一乐器。共 **4 类**（CTIS 端 '唢呐2'
  经 `display_cname` 规范化后对齐）。
- **同族不同型号**：两组——(1) 阮族：CTIS 的“大阮” vs ChMusic 的“中阮”；
  (2) 笙族：CTIS 的“中音笙” vs ChMusic 的“笙”（型号未注明）。本书**均不视为同类**
  （形制、音域差异明显，等同会让分类问题失真）。
- **仅 CTIS 有**：扬剧主胡、低音热瓦普、民间热瓦普、八角月琴。
- **仅 ChMusic 有**：三弦、笛子、坠琴、古筝、扬琴。


In [ ]:
# CTIS top-10 类名（按 default_train 样本数降序）
ctis_top10_display = [
    c['cname'] if c['cname'] != '唢呐2' else '唢呐 (原 CTIS cname="唢呐2")'
    for c in top10
]

rows = []
ctis_names_norm = {('唢呐' if c['cname'] == '唢呐2' else c['cname']) for c in top10}
for chm_name in order_11:
    if chm_name in ctis_names_norm:
        status = '严格交集'
    elif chm_name == '中阮':
        status = '同族异型（大阮）— 不视为同类'
    elif chm_name == '笙':
        status = '同族异型（中音笙）— 不视为同类'
    else:
        status = '仅 ChMusic 有'
    rows.append({'ChMusic 类名': chm_name, '状态': status,
                 'ChMusic 切片数': clip_counts[chm_name]})

df_align = pd.DataFrame(rows)
df_align


### 8.3 原始 top-10 四类交集的Mel谱示例

每类取第一个文件的首个切片绘制log-Mel谱。单张谱图可展示该片段的动态范围和
时频结构，但不能据此估计两个数据集的总体差异，也不能解释模型的外部测试结果。


In [ ]:
import librosa
import librosa.display

def _logmel(audio, sr, n_mels=64, n_fft=2048, hop=512):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop
    )
    return librosa.power_to_db(mel, ref=np.max)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
for ax, cname in zip(axes, CTIS_INTERSECTION):
    sample = next(c for c in chm_all if c['cname'] == cname and c['clip_idx'] == 0)
    S = _logmel(sample['audio'], sample['sr'])
    img = librosa.display.specshow(
        S, sr=sample['sr'], hop_length=512, x_axis='time', y_axis='mel', ax=ax,
        cmap='gray_r',
    )
    ax.set_xlabel('时间（秒）')
    ax.set_ylabel('频率（Hz）')
    ax.set_title(f'ChMusic · {cname}')
fig.colorbar(img, ax=axes, format='%+2.0f dB', fraction=0.015, pad=0.02)
fig.savefig(FIG_DIR / 'chmusic_intersection_melspec.png', dpi=600, bbox_inches='tight')
plt.show()


### 8.4 元数据与解释边界

本地副本可确认每个标签有5个音频文件，但未附演奏者和录音设备元数据。因此：

- ChMusic结果只能描述这一独立数据来源上的表现，不能代表所有外部环境；
- 现有文件不足以把内外差异分解为演奏者、设备、房间、曲目或其他因素；
- 若要检验具体因素，需要补充相应元数据或建立控制其他条件的对照数据。


## 9. 清洗后的数据集：实验所用的最终契约

前面1--8节检查原始标签和旧版探索切片。6.1b--6.4的主实验不直接使用这些临时
输出，而是统一读取 `_data_pipeline/output/` 中的冻结manifest。该管线处理族级标签
映射、录音组去重、静音与能量筛查、重采样、切片和组感知划分；具体规则及例外见
`freeze_config.yaml`、`cleaning_standard_v3.md`和审计manifest。

冻结输出的主要契约为：

- **6 类族级标签**：二胡 / 琵琶 / 中阮 / 笛子 / 唢呐 / 笙
- **24 kHz、5 秒非重叠切片**（活跃帧比例 ≥ 0.2，过滤静音）
- **Chromaprint 指纹 + SHA256 双重去重**，组感知 split
- **三份冻结的 train / val / test 划分**（底层种子为3407、2027、9413）
- **ChMusic 单独作为 external_test**，从契约层就不参与训练

下面读 `segment_manifest_seed0.csv` 与 `frozen_summary.json`，把这份契约可视化。


### 9.0 如何运行数据清理流程

如果 `_data_pipeline/output/` 还不存在，按下面 6 步跑一遍。
所有命令都从**本仓库根目录**（含 `CODE/` 文件夹的那一层）执行；
Windows 用 PowerShell 时把 `/` 换成 `\`。

#### 1. 下载两份原始数据集

| 数据集 | 来源 | 角色 |
|---|---|---|
| CCMusic CTIS | [HuggingFace `ccmusic-database/CTIS`](https://huggingface.co/datasets/ccmusic-database/CTIS) | train / val / test 主体 |
| ChMusic | https://github.com/haoranweiUTS/ChMusic | external_test |

CCMusic 是 HuggingFace Arrow 格式（音频字节嵌入在 Arrow 文件里）；
ChMusic 是 `.wav`。两者各遵循原始许可证。

#### 2. 把 CCMusic 转成 wav，并导出元数据

清洗流程不读 Arrow，先用 `preprocess/extract_ctis_audio.py` 抽取音频，
得到 `<某处>/ccmusic_ctis_audio/<乐器名>/*.wav`。
同时用 `preprocess/export_ctis_metadata.py` 导出元数据 CSV，**本 notebook 第 5 节**
做跨 split 重叠检测要用到。

```bash
pip install pyarrow   # 这两个脚本的唯一额外依赖

# 抽取音频字节为 wav（替换 /path/to/CCMUSIC_CTIS 为你下载到的本地路径）
python CODE/chapter06/_data_pipeline/preprocess/extract_ctis_audio.py \
  --dataset-root /path/to/CCMUSIC_CTIS \
  --output-dir   /path/to/CCMUSIC_CTIS/ccmusic_ctis_audio

# 导出元数据（4 个 split 的 audio_sha1 等字段）
python CODE/chapter06/_data_pipeline/preprocess/export_ctis_metadata.py \
  --dataset-root /path/to/CCMUSIC_CTIS \
  --output-dir   CODE/chapter06/_data_pipeline/preprocess/metadata_export
```

> 第二条命令把元数据放到固定位置 `_data_pipeline/preprocess/metadata_export/`，
> 本 notebook 第 5 节就从这里读。

#### 3. 安装清洗依赖

```bash
pip install numpy librosa soundfile soxr PyYAML pandas
```

系统命令 `fpcalc`（Chromaprint，去重用）：

| 系统 | 命令 |
|---|---|
| macOS | `brew install chromaprint` |
| Ubuntu / Debian | `sudo apt install libchromaprint-tools` |
| Windows | `choco install chromaprint`，或从 [acoustid.org](https://acoustid.org/chromaprint) 下载并加入 PATH |

验证：`fpcalc -v`，应 ≥ 1.5。

#### 4. 写入配置

```bash
cp CODE/chapter06/_data_pipeline/config.yaml.example \
   CODE/chapter06/_data_pipeline/config.yaml
```

编辑 `CODE/chapter06/_data_pipeline/config.yaml`，把两个数据集路径填**绝对路径**：

```yaml
data_root_paths:
  CCMusic: /path/to/CCMUSIC_CTIS/ccmusic_ctis_audio   # 第 2 步抽出来的 wav 目录
  ChMusic: /path/to/ChMusic/Musics
```

`config.yaml` 已在 `.gitignore` 中。

#### 5. 跑清洗流程

```bash
python CODE/chapter06/_data_pipeline/pipeline.py \
       CODE/chapter06/_data_pipeline/config.yaml
```

5–20 分钟，产物写到 `CODE/chapter06/_data_pipeline/output/`。

#### 6. 校验

```bash
wc -l CODE/chapter06/_data_pipeline/output/segment_manifest_seed0.csv
# 应为 2103（含表头，2102 段）
```

若数量不一致，应依次核对原始数据版本、`freeze_config.yaml`、依赖版本、人工裁决文件
和管线日志，不能只根据最终行数判断原因。

#### 已在别处跑过？

通过环境变量复用，避免再跑一遍：

```bash
# macOS / Linux
export CTMP_OUTPUT_DIR=/absolute/path/to/output
jupyter notebook
```

```powershell
# Windows PowerShell
$env:CTMP_OUTPUT_DIR = "C:\absolute\path\to\output"
jupyter notebook
```

环境变量需要在启动 Jupyter **之前**设置，否则 kernel 读不到。


In [ ]:
import json
from chapter06._common.ctmp_loader import (
    CTMP_CLASSES,
    get_ctmp_output_dir,
    load_segment_manifest,
)

try:
    CTMP_OUT = get_ctmp_output_dir()
    HAS_CTMP = True
    print('CTMP 输出目录:', CTMP_OUT)
except FileNotFoundError as e:
    print('未找到清洗输出。请按上面 9.0 节的步骤先跑 pipeline，')
    print('或设置 CTMP_OUTPUT_DIR 指向已有 output/ 目录后重启 kernel。')
    print()
    print('详细错误:', e)
    HAS_CTMP = False


### 9.1 漏斗：从原始扫描到冻结切片

`frozen_summary.json`记录录音级保留数量、切片数量和剔除原因计数。下图分别展示管线各阶段的记录数和已登记的剔除原因。不同层级的计数单位并不相同，不能把
“原始扫描→5秒切片”解释为单一留存率。


In [ ]:
if HAS_CTMP:
    summary = json.loads((CTMP_OUT / 'frozen_summary.json').read_text(encoding='utf-8'))
    drop_counts = summary.get('drop_reason_counts', {})

    # 扫描总数：audit_manifest 行数（含被剔除的）；冻结样本：frozen_manifest 行数
    audit_df = pd.read_csv(CTMP_OUT / 'audit_manifest.csv')
    frozen_df = pd.read_csv(CTMP_OUT / 'frozen_manifest.csv')
    seg_df_seed0 = pd.read_csv(CTMP_OUT / 'segment_manifest_seed0.csv')

    funnel = {
        '原始扫描': len(audit_df),
        '冻结样本（录音级）': len(frozen_df),
        '5 秒切片（seed0）': len(seg_df_seed0),
    }
    print(funnel)
    print('drop_reason_counts:', drop_counts)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))
    names = list(funnel.keys()); vals = list(funnel.values())
    bars = ax1.barh(names, vals, color=['0.4', '0.55', '0.7'])
    for bar, v in zip(bars, vals):
        ax1.text(v, bar.get_y() + bar.get_height() / 2, f' {v}', va='center')
    ax1.set_xlabel('样本数'); ax1.set_title('清洗漏斗')
    ax1.set_xlim(right=max(vals) * 1.18)
    ax1.invert_yaxis()

    if drop_counts:
        items = sorted(drop_counts.items(), key=lambda kv: -kv[1])
        ax2.barh([k for k, _ in items], [v for _, v in items], color='0.5')
        for i, (_, v) in enumerate(items):
            ax2.text(v, i, f' {v}', va='center')
        ax2.set_xlabel('被剔除的样本数'); ax2.set_title('剔除原因分布')
        ax2.set_xlim(right=max(v for _, v in items) * 1.18)
        ax2.invert_yaxis()
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'ctmp_cleaning_funnel.png', dpi=600, bbox_inches='tight')
    plt.show()


### 9.2 每类 × 每 split 的切片数

训练集略有不均（笙 245 vs 琵琶 65），val/test 因为录音组数量少所以更稀疏，
 external_test 全部来自 ChMusic、与训练完全隔离。**注意**：琵琶在 test 仅 4 段，
单份冻结划分不能反映划分间波动；6.1b和6.2对三份冻结划分报告均值与总体标准差。


In [ ]:
if HAS_CTMP:
    pivot = (seg_df_seed0
             .groupby(['split', 'family_label']).size()
             .unstack(fill_value=0)
             .reindex(columns=list(CTMP_CLASSES))
             .reindex(['train', 'val', 'test', 'external_test']))
    print(pivot)

    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    im = ax.imshow(pivot.values, aspect='auto', cmap='gray_r')
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=0)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            color = 'white' if v > pivot.values.max() * 0.55 else 'black'
            ax.text(j, i, int(v), ha='center', va='center', color=color, fontsize=9)
    ax.set_title('seed 0：每类 × 每 split 的 5 秒切片数')
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'ctmp_per_class_per_split.png', dpi=600, bbox_inches='tight')
    plt.show()


### 9.3 切片质量：active_frame_ratio 分布

每个切片的 `active_frame_ratio` 是非静音帧占总帧数的比例。进入冻结集的切片均满足
当前阈值要求。箱线图描述各类切片的统计分布；类别差异可能同时受演奏内容、切片位置、
录音电平和筛查规则影响，不能仅由乐器发声方式解释。


In [ ]:
if HAS_CTMP:
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    data = [seg_df_seed0[seg_df_seed0['family_label'] == c]['active_frame_ratio'].values
            for c in CTMP_CLASSES]
    bp = ax.boxplot(data, tick_labels=list(CTMP_CLASSES), patch_artist=True,
                    medianprops=dict(color='black'))
    for patch in bp['boxes']:
        patch.set(facecolor='0.85', edgecolor='0.3')
    ax.axhline(0.2, color='0.4', linestyle='--', linewidth=0.8)
    ax.text(0.78, 0.21, ' 冻结阈值 0.2', color='0.4', fontsize=8,
            ha='left', va='bottom', transform=ax.get_yaxis_transform())
    ax.set_ylabel('active_frame_ratio')
    ax.set_title('每类切片活跃度分布（seed 0，全部 split）')
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'ctmp_active_frame_ratio.png', dpi=600, bbox_inches='tight')
    plt.show()


### 9.4 与原始数据的契约对照

| 维度 | CTIS 原始（前文 1–8 节） | 清洗后的 CTMP 切片（6.1b / 6.2 使用） |
|---|---|---|
| 类别 | 原始细粒度标签；前文另列top-10统计 | 6 类族级标签 |
| 样本单位 | 原始数据条目或曲目文件 | 5 秒非重叠切片 |
| 采样率 | 44.1 kHz（CTIS）/ 22.05 kHz（ChMusic） | 统一 24 kHz |
| 静音 / 余音 | 大量保留 | active_frame_ratio ≥ 0.2 |
| 去重 | 无 | SHA256 + Chromaprint 双重 + 组感知 split |
| 外部测试集 | 后期临时拼接 | 契约层定义（external_test） |
| 数据划分 | 原始数据自带split或临时探索集合 | 三份冻结划分 |

6.1b--6.4的实验均以右列契约为准。重跑结果不一致时，应同时核对
`frozen_summary.json`、segment manifest、训练配置、设备和结果缓存元数据。
